# Bootstrap SNN Training

The underlying principle for ANN-SNN conversion is that the ReLU activation function (or similar form) approximates the firing rate of an LIF spiking neuron. Consequently, an ANN trained with ReLU activation can be mapped to an equivalent SNN with proper scaling of weights and thresholds. However, as the number of time-steps reduces, the alignment between ReLU activation and LIF spiking rate falls apart mainly due to the following two reasons (especially, for discrete-in-time models like Loihi’s CUBA LIF):

* With less time steps, the SNN can assume only a few discrete firing rates.
* Limited time steps mean that the spiking neuron activity rate often saturates to maximum allowable firing rate.

Introducing __Bootstrap training__. An SNN is used to jumpstart an equivalent ANN model which is then used to accelerate SNN training. There is no restriction on the type of spiking neuron or it's reset behavior. It consists of following steps:
![](fit.png)

* Input output data points are first collected from the network running as an SNN: __SAMPLING mode__. 
* The data is used to estimate the corresponding ANN activation as a piecewise linear layer, unique to each layer: __FIT mode__.
* The training is accelerated using the piecewise linear ANN activation: __ANN mode__.
* The network is seamlessly translated to an SNN: __SNN mode__.
* _SAMPLING mode_ and _FIT mode_ are repeated for a few iterations every couple of epochs, thus maintaining an accurate ANN estimate.


<table><tr>
<td> <img src="bootstrap.png" alt="Drawing" style="height: 400px;"/> </td>
</tr>
</table>

Bootstrap training is available as __`lava.lib.dl.bootstrap`__. The main modules are 

* `block`: provides `lava.lib.dl.slayer.block` based network definition interface.
* `ann_sampler`: provides utilities for sampling SNN data points and pievewise linear ANN fit.
* `routine`: `routine.Scheduler` provides scheduling utility to seamlessly switch between SAMPLING | FIT | ANN | SNN mode.
    * It also provides ANN-SNN bootstrap hybrid traiing utility as well (Not demonstrated in this tutorial).
    
<table><tr>
<td> <img src="scheduler.png" alt="Drawing" style="height: 350px;"/> </td>
</tr>
</table>

## URMA Student Policy Regression

Here, we will demonstrate bootstrap SNN training on the URMA teacher-student DAgger dataset saved in `teacher_student_dagger_dataset.npz`.


In [3]:
import os, sys
import h5py
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split

# import slayer from lava-dl
import lava.lib.dl.slayer as slayer
import lava.lib.dl.bootstrap as bootstrap

import IPython.display as display
from matplotlib import animation

# Create Network

The network definition follows standard PyTorch way using `torch.nn.Module`.

`lava.lib.dl.bootstrap` provides __block interface__ similar to `lava.lib.dl.slayer.block` - which bundles all these individual components into a single unit. These blocks can be cascaded to build a network easily. The block interface provides additional utilities for normalization (weight and neuron), dropout, gradient monitoring and network export.

In [4]:
class Network(torch.nn.Module):
    def __init__(
        self,
        input_dim=668,
        output_dim=24,
        hidden_dims=(1024, 1024, 1024, 1024, 1024),
        time_steps=5,
        input_strategy='signed_split',
    ):
        super(Network, self).__init__()
        self.input_dim = int(input_dim)
        self.output_dim = int(output_dim)
        self.hidden_dims = tuple(int(dim) for dim in hidden_dims)
        self.time_steps = int(time_steps)
        self.input_strategy = input_strategy
        if self.input_strategy not in {'identity', 'signed_split'}:
            raise ValueError(f'Unsupported input_strategy: {self.input_strategy}')
        self.encoded_input_dim = self.input_dim * 2 if self.input_strategy == 'signed_split' else self.input_dim

        neuron_params = {
                'threshold'     : 0.5,
                'current_decay' : 0.3,
                'voltage_decay' : 0.02,
                'tau_grad'      : 1,
                'scale_grad'    : 1,
            }
        neuron_params_norm = {
                **neuron_params,
                # 'norm'    : slayer.neuron.norm.MeanOnlyBatchNorm,
            }

        blocks = [
                bootstrap.block.cuba.Input(neuron_params, weight=2.0, bias=0, delay_shift=False),
                bootstrap.block.cuba.Flatten(),
            ]
        prev_dim = self.encoded_input_dim
        for hidden_dim in self.hidden_dims:
            blocks.append(
                bootstrap.block.cuba.Dense(
                    neuron_params_norm,
                    prev_dim,
                    hidden_dim,
                    weight_norm=False,
                    weight_scale=1,
                    delay_shift=False,
                )
            )
            prev_dim = hidden_dim
        blocks.append(
            bootstrap.block.cuba.Affine(
                neuron_params,
                prev_dim,
                self.output_dim,
                weight_norm=False,
                weight_scale=1,
                dynamics=False,
            )
        )
        self.blocks = torch.nn.ModuleList(blocks)


    def initialize_from_ann_checkpoint(self, checkpoint_path, device=None):
        checkpoint = torch.load(checkpoint_path, map_location=device or 'cpu')
        state_dict = checkpoint['state_dict'] if isinstance(checkpoint, dict) and 'state_dict' in checkpoint else checkpoint
        weight_keys = sorted(
            [key for key in state_dict if key.startswith('net.') and key.endswith('.weight')],
            key=lambda key: int(key.split('.')[1]),
        )
        bootstrap_layers = [block for block in self.blocks if hasattr(block, 'synapse')]
        if len(weight_keys) != len(bootstrap_layers):
            raise ValueError(f'ANN has {len(weight_keys)} linear layers, bootstrap network has {len(bootstrap_layers)} weighted layers.')

        for layer_index, (weight_key, bootstrap_layer) in enumerate(zip(weight_keys, bootstrap_layers)):
            weight = state_dict[weight_key].detach().to(
                bootstrap_layer.synapse.weight.device,
                dtype=bootstrap_layer.synapse.weight.dtype,
            )
            if layer_index == 0 and self.input_strategy == 'signed_split':
                weight = torch.cat([weight, -weight], dim=1)
            weight = weight.reshape(weight.shape[0], weight.shape[1], 1, 1, 1)
            if weight.shape != bootstrap_layer.synapse.weight.shape:
                raise ValueError(f'{weight_key} has mapped shape {tuple(weight.shape)}, expected {tuple(bootstrap_layer.synapse.weight.shape)}.')
            bootstrap_layer.synapse.weight.data.copy_(weight)
        return self

    def encode_input(self, x):
        if x.ndim == 1:
            x = x.reshape(1, -1)
        if x.ndim > 2:
            x = x.reshape(x.shape[0], -1)
        if x.shape[1] != self.input_dim:
            raise ValueError(f'Expected {self.input_dim} student observation features, received {x.shape[1]}.')
        if self.input_strategy == 'identity':
            return x
        return torch.cat([torch.relu(x), torch.relu(-x)], dim=1)

    def forward(self, x, mode):
        x = self.encode_input(x)
        x = x.reshape(x.shape[0], self.encoded_input_dim, 1, 1, 1)

        if mode.base_mode != bootstrap.Mode.ANN and self.time_steps > 1:
            x = x.repeat(1, 1, 1, 1, self.time_steps)

        for block, m in zip(self.blocks, mode):
            x = block(x, mode=m)

        while x.ndim > 3 and x.shape[-2] == 1:
            x = x.squeeze(-2)
        if x.ndim == 2:
            x = x.unsqueeze(-1)
        return x

    def export_hdf5(self, filename):
        h = h5py.File(filename, 'w')
        simulation = h.create_group('simulation')
        simulation['Ts'] = 1
        simulation['tSample'] = self.time_steps
        layer = h.create_group('layer')
        for i, b in enumerate(self.blocks):
            b.export_hdf5(layer.create_group(f'{i}'))


# Instantiate Network, Optimizer, Dataset and DataLoader

This cell loads `teacher_student_dagger_dataset.npz`, where `states` are 668-dimensional student observations and `actions` are 24-dimensional teacher action targets.


In [6]:
trained_folder = 'Trained'
os.makedirs(trained_folder, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

LEARNING_RATE = 1e-3
TRAIN_BATCH_SIZE = 1024
VAL_BATCH_SIZE = 1024
VAL_EVAL_SAMPLES = 10_000
FULL_VAL_INTERVAL = 10
PIN_MEMORY = device.type == 'cuda'
LR_PLATEAU_FACTOR = 0.5
LR_PLATEAU_PATIENCE = 5
LR_PLATEAU_THRESHOLD = 1e-4
MIN_LEARNING_RATE = 1e-5

ann_checkpoint_path = 'student_model_latest.pth'
net = Network().to(device)
net.initialize_from_ann_checkpoint(ann_checkpoint_path, device=device)
print(f'Initialized bootstrap network from {ann_checkpoint_path}')

optimizer = torch.optim.Adam(net.parameters(), lr=LEARNING_RATE)
lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=LR_PLATEAU_FACTOR,
    patience=LR_PLATEAU_PATIENCE,
    threshold=LR_PLATEAU_THRESHOLD,
    min_lr=MIN_LEARNING_RATE,
)

class TeacherStudentDataset(Dataset):
    def __init__(self, states, actions):
        self.states = torch.as_tensor(states, dtype=torch.float32)
        self.actions = torch.as_tensor(actions, dtype=torch.float32)

    @classmethod
    def from_npz(cls, npz_file):
        data = np.load(npz_file)
        return cls(states=data['states'], actions=data['actions'])

    def __len__(self):
        return len(self.states)

    def __getitem__(self, idx):
        return self.states[idx], self.actions[idx]

# Dataset and DataLoader instances.
dataset_path = 'teacher_student_dagger_dataset.npz'
student_dataset = TeacherStudentDataset.from_npz(dataset_path)

val_split = 0.2
val_size = max(1, int(len(student_dataset) * val_split))
train_size = len(student_dataset) - val_size
training_set, testing_set = random_split(
    student_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(0),
)

val_eval_size = min(VAL_EVAL_SAMPLES, len(testing_set))
val_holdout_size = len(testing_set) - val_eval_size
if val_holdout_size > 0:
    val_eval_set, _ = random_split(
        testing_set,
        [val_eval_size, val_holdout_size],
        generator=torch.Generator().manual_seed(1),
    )
else:
    val_eval_set = testing_set

train_loader = DataLoader(dataset=training_set, batch_size=TRAIN_BATCH_SIZE, shuffle=True, pin_memory=PIN_MEMORY)
val_loader = DataLoader(dataset=val_eval_set, batch_size=VAL_BATCH_SIZE, shuffle=False, pin_memory=PIN_MEMORY)
full_val_loader = DataLoader(dataset=testing_set, batch_size=VAL_BATCH_SIZE, shuffle=False, pin_memory=PIN_MEMORY)

sample_state, sample_action = student_dataset[0]
print(f'Loaded {len(student_dataset)} samples from {dataset_path}')
print(f'state shape: {tuple(sample_state.shape)} action shape: {tuple(sample_action.shape)}')
print(f'train samples: {len(training_set)} validation samples: {len(testing_set)}')
print(f'per-epoch validation subset: {len(val_eval_set)} full validation interval: {FULL_VAL_INTERVAL}')
print(f'batch sizes: train={TRAIN_BATCH_SIZE} validation={VAL_BATCH_SIZE} learning_rate={LEARNING_RATE}')
print(
    f'lr plateau scheduler: factor={LR_PLATEAU_FACTOR} patience={LR_PLATEAU_PATIENCE} '
    f'threshold={LR_PLATEAU_THRESHOLD} min_lr={MIN_LEARNING_RATE}'
)

stats = slayer.utils.LearningStats()

# Use a fresh LayerMode for every forward pass because LayerMode is an iterator.
def pure_snn_rate(model, states):
    snn_mode = bootstrap.routine.LayerMode(0, bootstrap.Mode.SNN)
    output = model.forward(states, snn_mode)
    return torch.mean(output, dim=-1).reshape((states.shape[0], -1))


def evaluate_snn_mse(model, loader):
    model.eval()
    loss_sum = 0.0
    samples = 0
    with torch.no_grad():
        for states, actions in loader:
            states = states.to(device, non_blocking=PIN_MEMORY)
            actions = actions.to(device, non_blocking=PIN_MEMORY)
            rate = pure_snn_rate(model, states)
            loss = F.mse_loss(rate, actions)
            loss_sum += loss.item() * states.shape[0]
            samples += states.shape[0]
    return loss_sum / max(1, samples)


Initialized bootstrap network from student_model_latest.pth
Loaded 400000 samples from teacher_student_dagger_dataset.npz
state shape: (668,) action shape: (24,)
train samples: 320000 validation samples: 80000
per-epoch validation subset: 10000 full validation interval: 10
batch sizes: train=1024 validation=1024 learning_rate=0.001
lr plateau scheduler: factor=0.5 patience=5 threshold=0.0001 min_lr=1e-05


# Training Loop

The student policy is a continuous-action regressor, so optimization uses mean squared error between the decoded network output and the 24-dimensional teacher action target. Bootstrap `SAMPLE` and `FIT` batches are used only to prepare the ANN surrogate and are not counted as training loss.


In [ ]:
epochs = 100
best_snn_val_loss = float('inf')
train_snn_loss_history = []
val_snn_loss_history = []
full_val_snn_loss_history = []
learning_rate_history = []
output_activity_history = []

for epoch in range(epochs):
    net.train()
    train_snn_loss_sum = 0.0
    train_output_abs_sum = 0.0
    train_output_numel = 0
    train_samples = 0

    for states, actions in train_loader:
        states = states.to(device, non_blocking=PIN_MEMORY)
        actions = actions.to(device, non_blocking=PIN_MEMORY)

        rate = pure_snn_rate(net, states)
        loss = F.mse_loss(rate, actions)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_snn_loss_sum += loss.detach().item() * states.shape[0]
        train_output_abs_sum += rate.detach().abs().sum().item()
        train_output_numel += rate.numel()
        train_samples += states.shape[0]

    train_snn_loss = train_snn_loss_sum / max(1, train_samples)
    train_snn_loss_history.append(train_snn_loss)
    output_activity = train_output_abs_sum / max(1, train_output_numel)
    output_activity_history.append(output_activity)

    val_snn_loss = evaluate_snn_mse(net, val_loader)
    val_snn_loss_history.append(val_snn_loss)

    run_full_val = epoch == 0 or (epoch + 1) % FULL_VAL_INTERVAL == 0 or epoch == epochs - 1
    full_val_snn_loss = None
    checkpoint_note = ''
    if run_full_val:
        full_val_snn_loss = evaluate_snn_mse(net, full_val_loader)
        if full_val_snn_loss < best_snn_val_loss:
            best_snn_val_loss = full_val_snn_loss
            torch.save(net.state_dict(), trained_folder + '/network.pt')
            checkpoint_note = ' saved_best'

    scheduler_metric = full_val_snn_loss if full_val_snn_loss is not None else val_snn_loss
    lr_scheduler.step(scheduler_metric)
    current_lr = optimizer.param_groups[0]['lr']
    learning_rate_history.append(current_lr)

    full_val_snn_loss_history.append(full_val_snn_loss)
    full_val_text = 'n/a' if full_val_snn_loss is None else f'{full_val_snn_loss:.9f}'

    print(
        f'[Epoch {epoch + 1:03d}/{epochs}] '
        f'train_snn_mse={train_snn_loss:.9f} val_subset_snn_mse={val_snn_loss:.9f} '
        f'full_val_snn_mse={full_val_text} best_snn_val_mse={best_snn_val_loss:.9f} '
        f'lr={current_lr:.2e} output_abs_mean={output_activity:.6f}{checkpoint_note}'
    )


[Epoch 001/100] train_snn_mse=0.054276732 val_subset_snn_mse=0.040286927 full_val_snn_mse=0.040704855 best_snn_val_mse=0.040704855 lr=1.00e-03 output_abs_mean=0.293905 saved_best
[Epoch 002/100] train_snn_mse=0.037879951 val_subset_snn_mse=0.035970006 full_val_snn_mse=n/a best_snn_val_mse=0.040704855 lr=1.00e-03 output_abs_mean=0.304799
[Epoch 003/100] train_snn_mse=0.035549745 val_subset_snn_mse=0.034785327 full_val_snn_mse=n/a best_snn_val_mse=0.040704855 lr=1.00e-03 output_abs_mean=0.305614
[Epoch 004/100] train_snn_mse=0.034507039 val_subset_snn_mse=0.034393658 full_val_snn_mse=n/a best_snn_val_mse=0.040704855 lr=1.00e-03 output_abs_mean=0.306840
[Epoch 005/100] train_snn_mse=0.033678914 val_subset_snn_mse=0.033443166 full_val_snn_mse=n/a best_snn_val_mse=0.040704855 lr=1.00e-03 output_abs_mean=0.308107
[Epoch 006/100] train_snn_mse=0.033017205 val_subset_snn_mse=0.031996621 full_val_snn_mse=n/a best_snn_val_mse=0.040704855 lr=1.00e-03 output_abs_mean=0.308480
[Epoch 007/100] train

# Plot the learning curves

Plot the train and validation MSE from the regression training loop.


In [ ]:
fig, loss_ax = plt.subplots(figsize=(10, 4))
loss_ax.plot(train_snn_loss_history, label='train SNN MSE')
loss_ax.plot(val_snn_loss_history, label='validation subset SNN MSE')
full_val_epochs = [index for index, value in enumerate(full_val_snn_loss_history) if value is not None]
full_val_losses = [value for value in full_val_snn_loss_history if value is not None]
loss_ax.plot(full_val_epochs, full_val_losses, marker='o', linestyle='none', label='full validation SNN MSE')
loss_ax.set_xlabel('Epoch')
loss_ax.set_ylabel('SNN MSE loss')
loss_ax.grid(True, alpha=0.3)

lr_ax = loss_ax.twinx()
lr_ax.plot(learning_rate_history, color='tab:gray', linestyle='--', alpha=0.7, label='learning rate')
lr_ax.set_ylabel('Learning rate')
lr_ax.set_yscale('log')

lines, labels = loss_ax.get_legend_handles_labels()
lr_lines, lr_labels = lr_ax.get_legend_handles_labels()
loss_ax.legend(lines + lr_lines, labels + lr_labels, loc='best')
fig.tight_layout()
plot_path = os.path.join(trained_folder, 'snn_training_curves.png')
fig.savefig(plot_path, dpi=200, bbox_inches='tight')
plt.close(fig)
print(f'Saved training plot to {plot_path}')


# Export the best model

Load the best model during training and export it as hdf5 network. It is supported by `lava.lib.dl.netx` to automatically load the network as a lava process.

In [ ]:
net.load_state_dict(torch.load(trained_folder + '/network.pt', map_location=device))
net.export_hdf5(trained_folder + '/network.net')


# Visualize the network output

Here, we will use `slayer.io.tensor_to_event` method to convert the torch output spike tensor into graded (non-binary) `slayer.io.Event` object and visualize a few input and output event pairs.

In [ ]:
output = net(input.to(device), mode=scheduler.mode(100, 0, False))
for i in range(5):
    img = (2*input[i].reshape(28, 28).cpu().data.numpy()-1) * 255
    Image.fromarray(img).convert('RGB').save(f'gifs/inp{i}.png')
    out_event = slayer.io.tensor_to_event(output[i].cpu().data.numpy().reshape(1, 10, -1))
    out_anim = out_event.anim(plt.figure(figsize=(10, 3.5)), frame_rate=2400)
    out_anim.save(f'gifs/out{i}.gif', animation.PillowWriter(fps=24), dpi=300)


In [ ]:
img_td = lambda gif: f'<td> <img src="{gif}" alt="Drawing" style="height: 150px;"/> </td>'
html = '<table>'
html += '<tr><td align="center"><b>Input</b></td><td><b>Output</b></td></tr>'
for i in range(5):
    html += '<tr>'
    html += img_td(f'gifs/inp{i}.png')
    html += img_td(f'gifs/out{i}.gif')
    html += '</tr>'
html += '</tr></table>'
display.HTML(html)